# Prompt Evaluation — fri-text till strukturerad golfstatistik

Jämför prompt-varianter på tre mått:
- **parse_rate** — returnerar modellen giltig JSON?
- **field_accuracy** — stämmer fältvärdena mot facit?
- **hallucination_rate** — innehåller svaret fält som inte ska finnas?

Varje variant körs mot samma facit (10 golfyttranden). Resultatet avgör vilket prompt-format vi går vidare med.

In [ ]:
import json
import re
import pandas as pd
from transformers import pipeline

print('Laddar modell...')
llm = pipeline('text-generation', model='HuggingFaceTB/SmolLM2-135M-Instruct')
print('Klar.')

In [ ]:
# Facit — 10 yttranden med förväntad JSON-output
# Fält: club, fairway_hit (0/1), gir (0/1), putts (int), lie (str)
# None = fältet är ej relevant för detta yttrande och ska inte förekomma i svaret

CASES = [
    {
        'utterance': 'Drive höger sida, landade i ruffen.',
        'expected': {'club': 'driver', 'fairway_hit': 0, 'lie': 'rough'},
        'context': None,
    },
    {
        'utterance': 'Rakt ner i fairway, perfekt position.',
        'expected': {'club': 'driver', 'fairway_hit': 1, 'lie': 'fairway'},
        'context': None,
    },
    {
        'utterance': 'Järn 7 mot greenen, landade tre meter från flaggan.',
        'expected': {'club': 'iron', 'gir': 1},
        'context': {'shot_no': 2, 'lie': 'fairway'},
    },
    {
        'utterance': 'Missade greenen till vänster, låg i ruffen.',
        'expected': {'club': 'iron', 'gir': 0, 'lie': 'rough'},
        'context': {'shot_no': 2, 'lie': 'fairway'},
    },
    {
        'utterance': 'Lågchip mot flaggan, stannade en meter bort.',
        'expected': {'club': 'wedge', 'gir': 0},
        'context': {'shot_no': 3, 'lie': 'rough'},
    },
    {
        'utterance': 'Hoppade över flaggan och rullade ut till två meter.',
        'expected': {'club': 'wedge', 'gir': 0},
        'context': {'shot_no': 3, 'lie': 'rough'},
    },
    {
        'utterance': 'Sank den på första puttet.',
        'expected': {'putts': 1},
        'context': {'shot_no': 4, 'lie': 'green'},
    },
    {
        'utterance': 'Tre puttar, sköt förbi två gånger.',
        'expected': {'putts': 3},
        'context': {'shot_no': 4, 'lie': 'green'},
    },
    {
        'utterance': 'Landade i bunkern vid greenen.',
        'expected': {'gir': 0, 'lie': 'bunker'},
        'context': {'shot_no': 2, 'lie': 'fairway'},
    },
    {
        'utterance': 'Två puttar, första nära men missade.',
        'expected': {'putts': 2},
        'context': {'shot_no': 5, 'lie': 'green'},
    },
]

print(f'{len(CASES)} testfall laddade.')

In [ ]:
# Prompt-varianter
# Varje variant är en funktion (utterance, context) -> str

def variant_minimal(utterance, context=None):
    return f'Extrahera golfslag: "{utterance}"\nJSON:'


def variant_schema(utterance, context=None):
    return (
        'Fält: club, fairway_hit (0/1), gir (0/1), putts, lie.\n'
        f'Yttrande: "{utterance}"\n'
        'JSON:'
    )


def variant_context(utterance, context=None):
    if context:
        ctx_str = f'Slag {context["shot_no"]}, lge: {context["lie"]}.\n'
    else:
        ctx_str = 'Slag 1 från tee.\n'
    return (
        ctx_str
        + f'Spelare: "{utterance}"\n'
        + 'Extrahera JSON:'
    )


VARIANTS = {
    'minimal':  variant_minimal,
    'schema':   variant_schema,
    'context':  variant_context,
}

# Förhandsgranska en prompt per variant
sample = CASES[0]
for name, fn in VARIANTS.items():
    print(f'--- {name} ---')
    print(fn(sample['utterance'], sample['context']))
    print()

In [ ]:
def extract_json(raw: str) -> dict | None:
    """Plockar ut första JSON-objektet ur råtexten. Returnerar None om inget hittades."""
    match = re.search(r'\{[^{}]+\}', raw)
    if not match:
        return None
    try:
        return json.loads(match.group())
    except json.JSONDecodeError:
        return None


def score(predicted: dict | None, expected: dict) -> dict:
    if predicted is None:
        return {'parsed': False, 'correct_fields': 0, 'total_fields': len(expected), 'hallucinated': 0}
    correct = sum(1 for k, v in expected.items() if predicted.get(k) == v)
    hallucinated = sum(1 for k in predicted if k not in expected)
    return {
        'parsed': True,
        'correct_fields': correct,
        'total_fields': len(expected),
        'hallucinated': hallucinated,
    }


# Kör alla varianter
rows = []
for variant_name, prompt_fn in VARIANTS.items():
    print(f'Kör variant: {variant_name} ...', flush=True)
    for i, case in enumerate(CASES):
        prompt = prompt_fn(case['utterance'], case['context'])
        out = llm([{'role': 'user', 'content': prompt}], max_new_tokens=60)
        raw = out[0]['generated_text'][-1]['content']
        parsed = extract_json(raw)
        s = score(parsed, case['expected'])
        rows.append({
            'variant': variant_name,
            'case': i,
            'utterance': case['utterance'],
            'raw': raw,
            'parsed_json': parsed,
            **s,
        })
    print('  klar')

df = pd.DataFrame(rows)
print('\nFärdigt.')

In [ ]:
# Aggregerad poäng per variant
summary = (
    df.groupby('variant')
    .agg(
        parse_rate=('parsed', 'mean'),
        field_accuracy=('correct_fields', lambda x: x.sum() / df.loc[x.index, 'total_fields'].sum()),
        hallucination_rate=('hallucinated', 'mean'),
        n=('case', 'count'),
    )
    .round(3)
    .reset_index()
)
print(summary.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

metrics = ['parse_rate', 'field_accuracy', 'hallucination_rate']
x = range(len(summary))
width = 0.25

fig, ax = plt.subplots(figsize=(9, 5))
for i, metric in enumerate(metrics):
    offset = (i - 1) * width
    bars = ax.bar([xi + offset for xi in x], summary[metric], width, label=metric)
    ax.bar_label(bars, fmt='%.2f', padding=2, fontsize=8)

ax.set_xticks(list(x))
ax.set_xticklabels(summary['variant'])
ax.set_ylim(0, 1.15)
ax.set_ylabel('Andel (0–1)')
ax.set_title('Prompt-varianter: parse_rate / field_accuracy / hallucination_rate')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Rådata — ett svar per fall för manuell granskning
for variant_name in VARIANTS:
    print(f'\n=== {variant_name.upper()} ===')
    subset = df[df['variant'] == variant_name][['case', 'utterance', 'parsed_json', 'correct_fields', 'total_fields', 'hallucinated']]
    print(subset.to_string(index=False))